In [3]:
import os
import requests

BASE = "https://api.semanticscholar.org/graph/v1"
ENDPOINT = f"{BASE}/paper/search"

# Optional: set an API key for higher rate limits (if you have one)
# In PowerShell: $env:S2_API_KEY="YOUR_KEY"
API_KEY = os.getenv("S2_API_KEY")

headers = {}
if API_KEY:
    headers["x-api-key"] = API_KEY

params = {
    "query": "graph neural networks dynamic graphs",
    "limit": 5,
    # fields is a comma-separated string in Semantic Scholar API
    "fields": "paperId,title,year,venue,abstract,url,authors,citationCount,openAccessPdf"
}

r = requests.get(ENDPOINT, params=params, headers=headers, timeout=30)
print("STATUS:", r.status_code)
r.raise_for_status()

data = r.json()

print("\nTOP-LEVEL KEYS:", list(data.keys()))
print("TOTAL (if provided):", data.get("total"))
print("NEXT OFFSET (if provided):", data.get("next"))

papers = data.get("data", [])
print("\nNUM PAPERS RETURNED:", len(papers))

if papers:
    p0 = papers[0]
    print("\nFIRST PAPER KEYS:", list(p0.keys()))
    print("\nFIRST PAPER SAMPLE:")
    print("paperId:", p0.get("paperId"))
    print("title:", p0.get("title"))
    print("year:", p0.get("year"))
    print("venue:", p0.get("venue"))
    print("citationCount:", p0.get("citationCount"))
    print("url:", p0.get("url"))
    print("openAccessPdf:", p0.get("openAccessPdf"))
    abs_ = (p0.get("abstract") or "")
    print("abstract (first 300 chars):", abs_[:300].replace("\n"," "))


STATUS: 429


HTTPError: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=graph+neural+networks+dynamic+graphs&limit=5&fields=paperId%2Ctitle%2Cyear%2Cvenue%2Cabstract%2Curl%2Cauthors%2CcitationCount%2CopenAccessPdf

In [6]:
import requests

BASE = "https://api.openalex.org/works"
params = {
    "search": "graph neural networks dynamic graphs",
    "per-page": 5,
    "select": ",".join([
        "id",
        "doi",
        "display_name",
        "publication_year",
        "abstract_inverted_index",
        "cited_by_count",
        "primary_location",
        "best_oa_location",
        "type",
        "authorships",
        "concepts",
        "counts_by_year"
    ])
}


r = requests.get(BASE, params=params, timeout=30)
print("STATUS:", r.status_code)
r.raise_for_status()
data = r.json()

print("TOP-LEVEL KEYS:", list(data.keys()))
results = data.get("results", [])
print("NUM RESULTS:", len(results))

def inverted_index_to_text(inv):
    # OpenAlex stores abstracts as {word: [positions]}
    # This reconstructs a readable abstract string when available.
    if not inv:
        return None
    pos_to_word = {}
    for w, positions in inv.items():
        for p in positions:
            pos_to_word[p] = w
    return " ".join(pos_to_word[p] for p in sorted(pos_to_word))

if results:
    w0 = results[0]
    print("\nFIRST WORK KEYS:", list(w0.keys()))
    print("\nFIRST WORK SAMPLE:")
    print("id:", w0.get("id"))
    print("title:", w0.get("display_name"))
    print("year:", w0.get("publication_year"))
    hv = w0.get("host_venue") or {}
    loc = w0.get("primary_location") or {}
    source = loc.get("source") or {}
    print("venue/source:", source.get("display_name"))
    print("landing_page:", loc.get("landing_page_url"))
    print("best_oa_pdf:", (w0.get("best_oa_location") or {}).get("pdf_url"))

    print("cited_by_count:", w0.get("cited_by_count"))
    abstract = inverted_index_to_text(w0.get("abstract_inverted_index"))
    if abstract:
        print("abstract (first 300 chars):", abstract[:300])
    else:
        print("abstract: None (not provided for this record)")


STATUS: 200
TOP-LEVEL KEYS: ['meta', 'results', 'group_by']
NUM RESULTS: 5

FIRST WORK KEYS: ['id', 'doi', 'display_name', 'publication_year', 'abstract_inverted_index', 'cited_by_count', 'primary_location', 'best_oa_location', 'type', 'authorships', 'concepts', 'counts_by_year']

FIRST WORK SAMPLE:
id: https://openalex.org/W2790404832
title: EEG Emotion Recognition Using Dynamical Graph Convolutional Neural Networks
year: 2018
venue/source: IEEE Transactions on Affective Computing
landing_page: https://doi.org/10.1109/taffc.2018.2817622
best_oa_pdf: None
cited_by_count: 1315
abstract (first 300 chars): In this paper, a multichannel EEG emotion recognition method based on a novel dynamical graph convolutional neural networks (DGCNN) is proposed. The basic idea of the proposed EEG emotion recognition method is to use a graph to model the multichannel EEG features and then perform EEG emotion classif
